In [11]:
!pip install spacy scikit-learn pandas numpy PyPDF2
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     ------ --------------------------------- 2.1/12.8 MB 8.4 MB/s eta 0:00:02
     ------- -------------------------------- 2.4/12.8 MB 6.8 MB/s eta 0:00:02
     ------------- -------------------------- 4.2/12.8 MB 6.0 MB/s eta 0:00:02
     ------------------ --------------------- 6.0/12.8 MB 6.6 MB/s eta 0:00:02
     ------------------------ --------------- 7.9/12.8 MB 7.0 MB/s eta 0:00:01
     ------------------------------ --------- 9.7/12.8 MB 7.3 MB/s eta 0:00:01
     ----------------------------------- ---- 11.3/12.8 MB 7.3 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 7.4 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 7.1 MB/s  0:00:01
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
import PyPDF2
import spacy
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
# Load model
nlp = spacy.load("en_core_web_sm")

In [24]:
# Extract text
def extract_text(file_path):
    text = ""
    with open(file_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

In [25]:
# Preprocess
def preprocess(text):
    doc = nlp(text.lower())
    return " ".join([t.lemma_ for t in doc if not t.is_stop and not t.is_punct])

In [26]:
# Load resumes
folder = "resumes"
resumes = []
names = []

for file in os.listdir(folder):
    if file.endswith(".pdf"):
        path = os.path.join(folder, file)
        resumes.append(preprocess(extract_text(path)))
        names.append(file)

In [27]:
# Job Description
jd = "Looking for Python, SQL, Machine Learning, Data Analysis skills"
jd = preprocess(jd)

In [28]:
# TF-IDF
vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform([jd] + resumes)


In [29]:
# Similarity
scores = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()


In [31]:
# Results
df = pd.DataFrame({
    "Resume": names,
    "Score": scores * 100
})
df = df.sort_values(by="Score", ascending=False)
df["Rank"] = range(1, len(df)+1)

In [32]:
# Save
df.to_csv("resume_ranking.csv", index=False)
df

,Resume,Score,Rank
3,resume4.pdf,8.910886,1
2,resume3.pdf,6.495433,2
1,resume2.pdf,0.000000,3
0,resume1.pdf,0.000000,4
